# 03 — Visualise the Pareto Front

This notebook loads the coreset produced by notebook 02 and plots the Pareto front in objective space (MMD² vs Sinkhorn divergence). It highlights three canonical selections from the front: the knee, the best-MMD point, and the best-Sinkhorn point.

**Prerequisite:** notebook 02 has been run (produces `runs_out_quickstart/nsga2-vae-popsoft-k30-rep0/`).

**Runtime:** a few seconds.

**Functions demonstrated:**
- [`select_pareto_representatives`](../docs/api/optimization.md#coreset_selectionoptimizationselect_pareto_representatives)

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'coreset_selection').is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the Pareto front
exp_dir = REPO / 'runs_out_quickstart' / 'nsga2-vae-popsoft-k30-rep0'
with np.load(exp_dir / 'coreset.npz', allow_pickle=True) as data:
    X = data['X']
    F = data['F']
    objectives = list(data['objectives'])

print(f'{F.shape[0]} Pareto-front members, objectives = {objectives}')

## 1. Pick representatives from the front

The library function returns a dict `{name: row_index}` for every canonical representative (knee, best-mmd, best-sinkhorn, chebyshev). Notebook 02 already saved these to `representatives/*.npz`; here we compute them on the fly.

In [ ]:
from coreset_selection.optimization import select_pareto_representatives

reps = select_pareto_representatives(F, objectives=tuple(objectives))
for name, idx in reps.items():
    print(f'  {name:<18} row={idx:>3}  F={F[idx]}')

## 2. Plot the front

Each dot is one coreset on the front. Each axis is one objective. The knee is the balanced compromise; best-mmd and best-sinkhorn are the extremes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(F[:, 0], F[:, 1], c='lightgray', s=40, edgecolor='k', linewidth=0.3, label='Pareto front')

markers = {'knee': ('o', 'tab:blue'), 'best-mmd': ('s', 'tab:green'), 'best-sinkhorn': ('^', 'tab:orange')}
for name, (marker, color) in markers.items():
    if name in reps:
        idx = reps[name]
        ax.scatter(F[idx, 0], F[idx, 1], marker=marker, s=180, facecolor=color,
                   edgecolor='k', linewidth=1.2, label=name, zorder=5)

ax.set_xlabel(objectives[0])
ax.set_ylabel(objectives[1])
ax.set_title(f'Pareto front (k=30, {F.shape[0]} members)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. How different are the three representatives?

Compare how many points each pair of representatives has in common. Fewer overlaps means the front is covering diverse regions of the decision space.

In [ ]:
masks = {name: X[idx].astype(bool) for name, idx in reps.items() if name in ('knee', 'best-mmd', 'best-sinkhorn')}
names = list(masks)
for i, a in enumerate(names):
    for b in names[i+1:]:
        overlap = int((masks[a] & masks[b]).sum())
        print(f'{a:<13} ∩ {b:<13}: {overlap}/{int(masks[a].sum())} points in common')

## What's next

- [04_interpret_metrics.ipynb](./04_interpret_metrics.ipynb) — decode every column in `metrics.csv`.